In [4]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import random
import joblib

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def generate_player_data(number_of_samples):
    data = []
    labels = []

    def apply_bounds(stats):
        try:
            stats['kills'] = max(0, int(stats['kills']))
            stats['deaths'] = max(1, int(stats['deaths']))
            stats['accuracy'] = np.clip(stats['accuracy'], 1.0, 100.0)
            stats['headshot_pct'] = np.clip(stats['headshot_pct'], 0.0, 100.0)
            stats['shots_fired'] = max(50, int(stats['shots_fired']))
            stats['shots_hit'] = int(stats['shots_fired'] * (stats['accuracy'] / 100.0))
            stats['survival_rate'] = np.clip(stats['survival_rate'], 0.0, 100.0)
            stats['reaction_time'] = max(10, stats['reaction_time'])
            stats['dmg_per_match'] = max(0, stats['dmg_per_match'])
            stats['avg_kill_dist'] = max(0, stats['avg_kill_dist'])
        except Exception as error:
            print("Error has occurred while applying bounds:", error)

        return stats

    def generate_normal_player():
        stats = {
            'kills': np.random.normal(12, 4),
            'deaths': np.random.normal(15, 3),
            'accuracy': np.random.normal(25, 5),
            'headshot_pct': np.random.normal(20, 7),
            'dmg_per_match': np.random.normal(1500, 400),
            'survival_rate': np.random.normal(35, 10),
            'avg_kill_dist': np.random.normal(25, 8),
            'shots_fired': np.random.normal(400, 100),
            'reaction_time': np.random.normal(250, 30)
        }

        if random.random() < 0.05:
            try:
                stats['kills'] += 15
                stats['headshot_pct'] += 15
            except Exception as error:
                print("Error has occurred while generating normal player:", error)

        return stats

    def generate_experienced_player():
        stats = {
            'kills': np.random.normal(25, 6),
            'deaths': np.random.normal(10, 3),
            'accuracy': np.random.normal(40, 6),
            'headshot_pct': np.random.normal(45, 8),
            'dmg_per_match': np.random.normal(3000, 600),
            'survival_rate': np.random.normal(55, 12),
            'avg_kill_dist': np.random.normal(35, 10),
            'shots_fired': np.random.normal(550, 120),
            'reaction_time': np.random.normal(190, 20)
        }

        try:
            if random.random() < 0.05:
                stats['avg_kill_dist'] = np.random.normal(70, 15)
                stats['accuracy'] += 10
            elif random.random() < 0.05: stats['deaths'] += 10
        except Exception as error:
            print("Error has occurred while generating experienced player:", error)

        return stats

    def generate_cheater_player():
        cheat_style = np.random.choice(['blatant', 'closet', 'wallhack'])

        if cheat_style == 'blatant':
            return {
                'kills': np.random.normal(45, 5),
                'deaths': np.random.normal(3, 2),
                'accuracy': np.random.normal(75, 10),
                'headshot_pct': np.random.normal(85, 5),
                'dmg_per_match': np.random.normal(5000, 500),
                'survival_rate': np.random.normal(80, 10),
                'avg_kill_dist': np.random.normal(40, 15),
                'shots_fired': np.random.normal(600, 100),
                'reaction_time': np.random.normal(80, 15) # Anomaly
            }
        elif cheat_style == 'closet':
            return {
                'kills': np.random.normal(15, 4),
                'deaths': np.random.normal(14, 3),
                'accuracy': np.random.normal(28, 5),
                'headshot_pct': np.random.normal(75, 8), # Anomaly
                'dmg_per_match': np.random.normal(1600, 300),
                'survival_rate': np.random.normal(40, 10),
                'avg_kill_dist': np.random.normal(25, 8),
                'shots_fired': np.random.normal(400, 100),
                'reaction_time': np.random.normal(110, 10) # Anomaly
            }
        else:
            return {
                'kills': np.random.normal(30, 5),
                'deaths': np.random.normal(5, 2),
                'accuracy': np.random.normal(20, 5), # Anomaly
                'headshot_pct': np.random.normal(25, 5),
                'dmg_per_match': np.random.normal(3500, 400),
                'survival_rate': np.random.normal(70, 10),
                'avg_kill_dist': np.random.normal(55, 12),
                'shots_fired': np.random.normal(1000, 200), # Anomaly
                'reaction_time': np.random.normal(230, 30)
            }

    for index in range(number_of_samples):
        try:
            player_type = np.random.choice([0, 1, 2])

            if player_type == 0: raw_stats = generate_normal_player()
            elif player_type == 1: raw_stats = generate_experienced_player()
            else: raw_stats = generate_cheater_player()

            clean_stats = apply_bounds(raw_stats)

            data.append(
                [
                    clean_stats['kills'],
                    clean_stats['deaths'],
                    clean_stats['accuracy'],
                    clean_stats['headshot_pct'],
                    clean_stats['dmg_per_match'],
                    clean_stats['survival_rate'],
                    clean_stats['avg_kill_dist'],
                    clean_stats['shots_fired'],
                    clean_stats['shots_hit'],
                    clean_stats['reaction_time']
                ]
            )
            labels.append(player_type)
        except Exception as error:
            print(f"An error has occurred on lap #{index}:", error)

    columns = ['Kills', 'Deaths', 'Accuracy', 'HeadshotPercent', 'DamagePerMatch', 'SurvivalRate', 'AverageKillDistance', 'ShotsFired', 'ShotsHit', 'ReactionTime']

    return pd.DataFrame(data, columns=columns), np.array(labels)

data_frame, labels = generate_player_data(1000)






SEED = 122
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_names = ['Обычный', 'Сильный', 'Подозрительный']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    data_frame.values.astype(np.float32), labels,
    test_size=0.20, random_state=SEED, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25, random_state=SEED, stratify=y_train_val
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=device)


class AntiCheat(nn.Module):
    def __init__(self, activation_name):
        super().__init__()

        activations = {
            'ReLU': nn.ReLU,
            'Tanh': nn.Tanh,
            'LeakyReLU': lambda: nn.LeakyReLU(negative_slope=0.1),
            'SiLU': nn.SiLU,
            'Sigmoid': nn.Sigmoid
        }
        if activation_name not in activations:
            raise ValueError(f'Неизвестная функция активации: {activation_name}')

        make_activation = activations[activation_name]
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            make_activation(),
            nn.Dropout(0.10),
            nn.Linear(64, 32),
            make_activation(),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.network(x)


def calculate_accuracy(model, features, targets):
    model.eval()
    with torch.no_grad():
        predicted_classes = model(features).argmax(dim=1)
    return (predicted_classes == targets).float().mean().item()


def train_experiment(activation_name, max_epochs=250, patience=30):
    torch.manual_seed(SEED)
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=64, shuffle=True, generator=generator
    )

    model = AntiCheat(activation_name).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    best_val_accuracy = -1.0
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch_features, batch_targets in train_loader:
            batch_features = batch_features.to(device)
            batch_targets = batch_targets.to(device)

            optimizer.zero_grad()
            logits = model(batch_features)
            loss = criterion(logits, batch_targets)
            loss.backward()
            optimizer.step()

        val_accuracy = calculate_accuracy(model, X_val_tensor, y_val_tensor)
        if val_accuracy > best_val_accuracy + 1e-6:
            best_val_accuracy = val_accuracy
            best_epoch = epoch
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    model.to(device)
    train_accuracy = calculate_accuracy(
        model, X_train_tensor.to(device), y_train_tensor.to(device)
    )
    return model, train_accuracy, best_val_accuracy, best_epoch



experiment_results = []
trained_models = {}
for activation_name in ['ReLU', 'Tanh', 'LeakyReLU', 'SiLU', 'Sigmoid']:
    model, train_accuracy, val_accuracy, best_epoch = train_experiment(activation_name)
    trained_models[activation_name] = model
    experiment_results.append({
        'Активация': activation_name,
        'Train accuracy': train_accuracy,
        'Validation accuracy': val_accuracy,
        'Лучшая эпоха': best_epoch
    })

results_table = pd.DataFrame(experiment_results).sort_values(
    'Validation accuracy', ascending=False
).reset_index(drop=True)
print('Сравнение функций активации:')
print(results_table.to_string(index=False, formatters={
    'Train accuracy': '{:.2%}'.format,
    'Validation accuracy': '{:.2%}'.format
}))

best_activation = results_table.loc[0, 'Активация']
best_model = trained_models[best_activation]
best_model.eval()

joblib.dump(scaler, 'anticheat_scaler.joblib')
torch.save(best_model.state_dict(), 'best_anticheat_model.pth')
print(f'\n[INFO] scaler was saved as "anticheat_scaler.joblib"')
print(f'[INFO] The model`s weights were saved as "best_anticheat_model.pth"')

with torch.no_grad():
    test_logits = best_model(X_test_tensor)
    test_predictions = test_logits.argmax(dim=1).cpu().numpy()

test_accuracy = accuracy_score(y_test, test_predictions)
print(f'\nЛучшая активация по validation: {best_activation}')
print(f'Test accuracy лучшей модели: {test_accuracy:.2%}')
print('\nClassification report:')
print(classification_report(
    y_test, test_predictions, labels=[0, 1, 2],
    target_names=class_names, digits=3, zero_division=0
))
print('Confusion matrix (строки — истинный класс, столбцы — предсказанный):')
print(confusion_matrix(y_test, test_predictions, labels=[0, 1, 2]))


def predict_player(kills, deaths, accuracy, headshot_percent, damage_per_match,
                   survival_rate, average_kill_distance, shots_fired, shots_hit,
                   reaction_time):
    player = np.array([[
        kills, deaths, accuracy, headshot_percent, damage_per_match,
        survival_rate, average_kill_distance, shots_fired, shots_hit, reaction_time
    ]], dtype=np.float32)

    player_scaled = scaler.transform(player).astype(np.float32)
    player_tensor = torch.tensor(player_scaled, dtype=torch.float32, device=device)

    best_model.eval()
    with torch.no_grad():
        logits = best_model(player_tensor)

        probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]

    verdict_index = int(np.argmax(probabilities))
    result = ' | '.join(
        f'{name} — {probability * 100:.1f}%'
        for name, probability in zip(class_names, probabilities)
    )
    print(f'{result} → Вердикт: {class_names[verdict_index].upper()}')
    return {name: float(probability) for name, probability in zip(class_names, probabilities)}



example_prediction = predict_player(32, 7, 58, 72, 3900, 68, 48, 620, 360, 105)

Сравнение функций активации:
Активация Train accuracy Validation accuracy  Лучшая эпоха
     ReLU        100.00%             100.00%            40
     Tanh         99.83%             100.00%            41
LeakyReLU        100.00%             100.00%            40
     SiLU        100.00%             100.00%            41
  Sigmoid         98.83%              99.50%           137

[INFO] scaler was saved as "anticheat_scaler.joblib"
[INFO] The model`s weights were saved as "best_anticheat_model.pth"

Лучшая активация по validation: ReLU
Test accuracy лучшей модели: 99.00%

Classification report:
                precision    recall  f1-score   support

       Обычный      1.000     0.985     0.992        66
       Сильный      0.971     1.000     0.986        68
Подозрительный      1.000     0.985     0.992        66

      accuracy                          0.990       200
     macro avg      0.990     0.990     0.990       200
  weighted avg      0.990     0.990     0.990       200

Co